# 微调 Laya 做「好感度 + 草稿发送后果」判断（Kaggle T4 ×2）

改自 Laya 官方的 `laya_finetune_typed_decisions_2xT4_kaggle.ipynb`，训练脚本原样保留，只换了数据和评测。

**用前准备**
1. 在本地跑完 `data/run_all.sh`，得到 `data/dataset/train.jsonl`、`dev.jsonl`、`test.jsonl`。
2. 在 Kaggle 上新建一个 Dataset，名字 `laya-chat-data`，把这三个文件传上去，然后在本笔记本右侧 Add Input 加进来（路径会是 `/kaggle/input/laya-chat-data/`）。
3. Notebook options：Accelerator 选 **GPU T4 x2**，Internet **On**。

**底座选多语言权重**：中文在 ModernBERT 的英文分词器下一个字要占两三个 token，512 的上下文根本放不下十条消息；mmBERT 的分词器对中文正常，上下文 1024。`head_max_len=512 / max_len=1024` 和运行时 `laya_chat` 的设置一致。


## 1. Environment & Dual T4 GPU Check
Verify both T4 GPUs are detected.


In [ ]:
!nvidia-smi
import os, subprocess, torch

n_gpu = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | Visible GPUs: {n_gpu}")
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} ({p.total_memory / 1e9:.1f} GB)")

assert n_gpu >= 2, (
    f"Expected 2 GPUs, but detected {n_gpu}!\n"
    "Please switch your Kaggle Accelerator: on the right sidebar, go to Notebook options -> "
    "Accelerator -> select GPU T4 x2."
)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("Both T4 GPUs verified and ready for DDP training!")


## 2. Install Dependencies


In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pyarrow pandas scipy accelerate tabulate
import laya, transformers, datasets, torch
print("Laya version        :", laya.__version__)
print("Transformers version:", transformers.__version__)
print("PyTorch version     :", torch.__version__)


## 3. Download & Preprocess Data for DDP
We preprocess all 1,200 training cases into tokenized items and save them to disk so both DDP worker ranks can read them.


In [ ]:
import os, json, torch
from datasets import load_dataset
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.common import build_sequence, render_options, QTYPES

MODEL_ID = "convaiinnovations/laya"
SUBFOLDER = "multilingual"          # 也可以试 "typed-decisions"（英文分词器，中文会被截断）
import glob
_found = glob.glob("/kaggle/input/**/train.jsonl", recursive=True)
assert _found, "没找到 train.jsonl：右侧 Input 面板里还没有加数据集（+ Add Input → Your Datasets → laya-chat-data）"
DATA_DIR = os.path.dirname(_found[0])       # 不管数据集叫什么名字、有没有子目录，找到就用
print("data dir:", DATA_DIR)
HEAD_MAX_LEN, MAX_LEN = 512, 1024

root = snapshot_download(MODEL_ID, allow_patterns=[f"{SUBFOLDER}/*"])
model_dir = os.path.join(root, SUBFOLDER)
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
cfg_path = os.path.join(model_dir, "rl_agent_config.json")
with open(cfg_path) as f:
    cfg = json.load(f)
cfg["head_max_len"], cfg["max_len"] = HEAD_MAX_LEN, MAX_LEN
with open(cfg_path, "w") as f:          # 训练脚本从这里读 cfg，并把它写进产出目录
    json.dump(cfg, f, indent=2)
print("base:", model_dir, "| head_max_len", HEAD_MAX_LEN, "max_len", MAX_LEN)

ds_train = load_dataset("json", data_files={"train": f"{DATA_DIR}/train.jsonl"})["train"]
print("train rows:", len(ds_train))

def build_training_item(state, q, gold_q):
    t = q["type"]
    crit = q.get("criteria", {})
    if t == "choice":
        keys = list(crit.keys())
        target = [gold_q["probabilities"].get(k, 0.0) for k in keys]
    elif t == "noul":
        target = [gold_q["probabilities"].get("false", 0.5), gold_q["probabilities"].get("true", 0.5)]
    elif t == "score":
        n_levels = len(crit) if isinstance(crit, list) else 4
        target = [gold_q["probabilities"].get(str(i), 0.0) for i in range(n_levels)]
    s = sum(target)
    target = [v / s for v in target] if s > 0 else [1.0 / len(target)] * len(target)
    label = target.index(max(target))
    k = len(render_options({"t": t, "crit": crit}))
    seq, markers = build_sequence(tok, state, {"t": t, "ins": q["instructions"], "crit": crit}, cfg["max_len"], cfg["head_max_len"])
    if len(markers) != k:
        return None
    return {"ids": seq, "markers": markers, "qtype": QTYPES[t], "target": target, "label": label}

items, dropped = [], 0
for row in ds_train:
    state = json.loads(row["state"]); questions = json.loads(row["questions"]); gold = json.loads(row["gold"])
    for qid, q in questions.items():
        if qid in gold:
            it = build_training_item(state, q, gold[qid])
            if it: items.append(it)
            else: dropped += 1
print(f"Preprocessed {len(items)} training sequences, dropped {dropped}")
torch.save(items, "/kaggle/working/train_items.pt")


## 4. DDP Training Script (`train_ddp.py`)
We write the multi-GPU distributed RLCD training script using pure policy gradients with proper scoring rules and DDP gradient synchronization.


In [ ]:
%%writefile /kaggle/working/train_ddp.py
import os, sys, time, json, random, math
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from laya.common import build_model, proper_reward, QTYPES

def collate_train_batch(items, pad_id):
    n, L = len(items), max(len(it["ids"]) for it in items)
    kmax = max(len(it["markers"]) for it in items)
    ids = torch.full((n, L), pad_id, dtype=torch.long)
    att = torch.zeros((n, L), dtype=torch.long)
    mpos = torch.zeros((n, kmax), dtype=torch.long)
    mmask = torch.zeros((n, kmax), dtype=torch.bool)
    target = torch.zeros((n, kmax), dtype=torch.float32)
    for i, it in enumerate(items):
        ids[i, : len(it["ids"])] = torch.tensor(it["ids"])
        att[i, : len(it["ids"])] = 1
        k = len(it["markers"])
        mpos[i, :k] = torch.tensor(it["markers"])
        mmask[i, :k] = True
        target[i, : len(it["target"])] = torch.tensor(it["target"], dtype=torch.float32)
    return {
        "input_ids": ids,
        "attention_mask": att,
        "marker_pos": mpos,
        "marker_mask": mmask,
        "target": target,
        "qtype": torch.tensor([it["qtype"] for it in items]),
        "label": torch.tensor([it["label"] for it in items])
    }

def fit_one_temp(sel):
    if len(sel) < 10:
        return 1.0
    kmax = max(len(z) for z, _ in sel)
    Z = torch.full((len(sel), kmax), -1e4)
    T = torch.zeros((len(sel), kmax))
    for i, (z, t) in enumerate(sel):
        Z[i, :len(z)] = torch.tensor(z)
        T[i, :len(t)] = torch.tensor(t, dtype=torch.float32)
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    def closure():
        opt.zero_grad()
        loss = -(T * torch.log_softmax(Z / log_t.exp(), -1)).sum(-1).mean()
        loss.backward()
        return loss
    opt.step(closure)
    return float(torch.clamp(log_t.exp(), 0.1, 10.0).item())

def main():
    dist.init_process_group("nccl")
    rank = dist.get_rank()
    world_size = dist.get_world_size()
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    model_dir = sys.argv[1]
    output_dir = sys.argv[2]
    
    with open(os.path.join(model_dir, "rl_agent_config.json")) as f:
        cfg = json.load(f)
    cfg["gradient_checkpointing"] = True
    cfg["max_tokens_per_batch"] = 4096
    cfg["max_len"] = 1024
    cfg["head_max_len"] = 256

    tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
    model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
    
    weights = load_file(os.path.join(model_dir, "model.safetensors"))
    model.load_state_dict(weights, strict=True)
    
    model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.head_checkpointing = True
    model.to(device)
    model.train()

    ddp_model = DDP(model, device_ids=[local_rank], find_unused_parameters=True)
    
    all_items = torch.load("/kaggle/working/train_items.pt", weights_only=False)
    my_items = all_items[rank::world_size]
    
    EPOCHS = 4
    MICRO_BATCH = 8      # 8 sequences per forward pass per GPU
    GRAD_ACCUM = 4       # Effective batch across 2 GPUs = 64 sequences (8 * 2 * 4)
    GROUP_SIZE = 4       # GRPO baseline samples
    LR_ENCODER = 2.5e-5  # Encoder adaptation rate
    LR_HEAD = 1.0e-4     # Head adaptation rate
    SIGMA_START = 0.4    # Exploration noise
    SIGMA_END = 0.1

    enc_params = [p for n, p in ddp_model.named_parameters() if "encoder." in n]
    head_params = [p for n, p in ddp_model.named_parameters() if "encoder." not in n]
    
    optimizer = torch.optim.AdamW([
        {"params": enc_params, "lr": LR_ENCODER},
        {"params": head_params, "lr": LR_HEAD}
    ], weight_decay=0.01)
    
    total_updates = (len(my_items) // (MICRO_BATCH * GRAD_ACCUM)) * EPOCHS
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, total_updates), eta_min=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=True)
    
    if rank == 0:
        print(f"Starting 2xT4 DDP training: {len(all_items)} total items | {len(my_items)} per rank | {EPOCHS} epochs")
    t0 = time.time()
    
    for epoch in range(EPOCHS):
        random.seed(42 + epoch + rank)
        random.shuffle(my_items)
        epoch_loss, n_batches = 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        accum_step = 0
        
        progress = epoch / max(1, EPOCHS - 1)
        sigma = SIGMA_START + (SIGMA_END - SIGMA_START) * progress
        
        for b_idx in range(0, len(my_items), MICRO_BATCH):
            chunk = my_items[b_idx:b_idx + MICRO_BATCH]
            if not chunk:
                continue
            
            batch = collate_train_batch(chunk, tok.pad_token_id)
            
            with torch.autocast("cuda", dtype=torch.float16):
                logits, act = ddp_model(
                    batch["input_ids"].to(device),
                    batch["attention_mask"].to(device),
                    batch["marker_pos"].to(device),
                    batch["marker_mask"].to(device),
                    batch["qtype"].to(device)
                )
            
            logits = logits.float()
            mask = batch["marker_mask"].to(device)
            k = mask.sum(-1, keepdim=True).float()
            target = batch["target"].to(device)
            
            # 1. Sample G noisy logit distributions with zero-mean projection
            eps = torch.randn((GROUP_SIZE,) + logits.shape, device=device) * sigma * mask
            eps = (eps - eps.sum(-1, keepdim=True) / k) * mask
            z = logits.detach().unsqueeze(0) + eps
            q = torch.softmax(z.masked_fill(~mask, -1e4), -1)
            
            # 2. Evaluate proper scoring reward (w_sph=0.75 for soft target matching)
            with torch.no_grad():
                r = proper_reward(q, target.unsqueeze(0), batch["qtype"].to(device), mask, w_sph=0.75, w_rps=1.0)
                adv = r - r.mean(0, keepdim=True)
                adv = adv / (adv.std() + 1e-6)
            
            # 3. Policy gradient loss + full 1.0 soft cross-entropy guidance
            logp = -(((z - logits.unsqueeze(0)) ** 2) * mask).sum(-1) / (2 * sigma ** 2)
            loss_rl = -(adv * logp).mean()
            loss_ce = -(target * torch.log_softmax(logits.masked_fill(~mask, -1e4), -1)).sum(-1).mean()
            loss = (loss_rl + 1.0 * loss_ce) / GRAD_ACCUM + 0.0 * act.sum()
            
            scaler.scale(loss).backward()
            accum_step += 1
            
            if accum_step % GRAD_ACCUM == 0 or (b_idx + MICRO_BATCH) >= len(my_items):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(ddp_model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)
            
            epoch_loss += loss.item() * GRAD_ACCUM
            n_batches += 1
            
            if rank == 0 and (n_batches % 50) == 0:
                cur_lr = scheduler.get_last_lr()[0]
                print(f"  Epoch {epoch+1}/{EPOCHS} | Step {n_batches} | Loss: {loss.item()*GRAD_ACCUM:.4f} | Reward: {r.mean().item():.3f} | LR: {cur_lr:.2e}")

        if rank == 0:
            print(f"=== Epoch {epoch+1}/{EPOCHS} Completed in {time.time()-t0:.1f}s | Avg Loss: {epoch_loss/max(1, n_batches):.4f} ===")

        dist.barrier()

        # Overwrite a single rolling checkpoint after each epoch so a crash,
        # OOM, or Kaggle session timeout doesn't lose all prior training.
        if rank == 0:
            ckpt_dir = os.path.join(output_dir, "checkpoint_latest")
            os.makedirs(ckpt_dir, exist_ok=True)
            ckpt_sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
            save_file(ckpt_sd, os.path.join(ckpt_dir, "model.safetensors"))
            model.encoder.config.save_pretrained(os.path.join(ckpt_dir, "encoder"))
            tok.save_pretrained(os.path.join(ckpt_dir, "tokenizer"))
            with open(os.path.join(ckpt_dir, "checkpoint_meta.json"), "w") as f:
                json.dump({
                    "epoch": epoch + 1,
                    "total_epochs": EPOCHS,
                    "avg_loss": epoch_loss / max(1, n_batches)
                }, f, indent=2)
            print(f"  Saved rolling checkpoint (epoch {epoch+1}/{EPOCHS}) to {ckpt_dir}")

    dist.barrier()
    
    # Post-training temperature calibration on rank 0 (micro-batched in chunks of 16 to prevent OOM)
    if rank == 0:
        print("\nFitting post-training calibration temperatures...")
        del optimizer, scaler, scheduler
        torch.cuda.empty_cache()
        model.eval()
        calib_items = all_items[::15][:400]
        calib_preds = []
        with torch.no_grad():
            for c_idx in range(0, len(calib_items), 16):
                c_chunk = calib_items[c_idx:c_idx + 16]
                cb = collate_train_batch(c_chunk, tok.pad_token_id)
                with torch.autocast("cuda", dtype=torch.float16):
                    l_sub, _ = model(
                        cb["input_ids"].to(device),
                        cb["attention_mask"].to(device),
                        cb["marker_pos"].to(device),
                        cb["marker_mask"].to(device),
                        cb["qtype"].to(device)
                    )
                l_np = l_sub.float().cpu().numpy()
                for r, it in enumerate(c_chunk):
                    k = len(it["markers"])
                    calib_preds.append((it["qtype"], l_np[r, :k], it["target"]))
        
        fitted_temps = [1.2, 1.2, 1.2]
        try:
            for qt in range(3):
                sel = [(z, t) for q_type, z, t in calib_preds if q_type == qt]
                if sel:
                    fitted_temps[qt] = fit_one_temp(sel)
            print("Fitted calibration temperatures (choice, score, noul):", [round(t, 3) for t in fitted_temps])
        except Exception as e:
            print("Temperature fitting fallback:", e)
        os.makedirs(output_dir, exist_ok=True)
        sd = {k: v.half().contiguous().cpu() for k, v in model.state_dict().items()}
        save_file(sd, os.path.join(output_dir, "model.safetensors"))
        model.encoder.config.save_pretrained(os.path.join(output_dir, "encoder"))
        tok.save_pretrained(os.path.join(output_dir, "tokenizer"))
        
        cfg["fine_tuned"] = True
        cfg["model_name"] = "laya-typed-decisions"
        cfg["temperature"] = fitted_temps
        with open(os.path.join(output_dir, "rl_agent_config.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        print(f"Model successfully saved to {output_dir}!")

    dist.destroy_process_group()

if __name__ == "__main__":
    main()


## 5. Launch Multi-GPU Fine-Tuning with `torchrun`
Runs on both T4 GPUs in parallel (~4 to 6 minutes total).


In [ ]:
import torch
OUTPUT_DIR = "/kaggle/working/laya_chat_finetuned"
MODEL_DIR = model_dir
NGPU = max(1, torch.cuda.device_count())      # 双 T4 就是 2；被分到单卡也能跑，只是慢一倍
cmd = f"torchrun --standalone --nproc_per_node={NGPU} /kaggle/working/train_ddp.py {MODEL_DIR} {OUTPUT_DIR}"
print(f"GPUs: {NGPU} | Executing:", cmd)
!{cmd}


## 6. 在 dev / test 上评测：微调前 vs 微调后

In [ ]:
import json, time
from collections import Counter, defaultdict
import laya

def evaluate(agent, path, name):
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    hit, tot, mae, majority = Counter(), Counter(), defaultdict(float), defaultdict(Counter)
    t0 = time.time()
    for r in rows:
        state, qs, gold = json.loads(r["state"]), json.loads(r["questions"]), json.loads(r["gold"])
        ans = agent.predict(state, qs)["answers"]
        for qid, q in qs.items():
            gp = gold[qid]["probabilities"]; want = max(gp, key=gp.get)
            majority[qid][want] += 1
            if q["type"] == "score":
                got = str(int(round(ans[qid]["score"])))
                mae[qid] += abs(ans[qid]["score"] - sum(int(k) * v for k, v in gp.items()))
            else:
                got = ans[qid]["choice"]
            hit[qid] += got == want; tot[qid] += 1
    print(f"\n=== {name}: {len(rows)} rows, {time.time() - t0:.0f}s ===")
    print(f"{'question':18} {'acc':>6} {'majority':>9}  note")
    for qid in tot:
        maj = majority[qid].most_common(1)[0][1] / tot[qid]
        note = f"level MAE={mae[qid] / tot[qid]:.2f}" if qid in mae else ""
        print(f"{qid:18} {hit[qid] / tot[qid]:6.2f} {maj:9.2f}  {note}")
    print(f"overall: {sum(hit.values()) / max(1, sum(tot.values())):.3f}")

agent_base = laya.Agent(MODEL_DIR, device="cuda")
agent_base.cfg["head_max_len"], agent_base.cfg["max_len"] = HEAD_MAX_LEN, MAX_LEN
evaluate(agent_base, f"{DATA_DIR}/dev.jsonl", "base / dev")
del agent_base
agent_ft = laya.Agent(OUTPUT_DIR, device="cuda")
evaluate(agent_ft, f"{DATA_DIR}/dev.jsonl", "fine-tuned / dev")
evaluate(agent_ft, f"{DATA_DIR}/test.jsonl", "fine-tuned / test")


## 7. 打包产出目录，下载后放到本机，把路径填进 `~/.laya-chat/config.json` 的 `model`（`subfolder` 留空）

In [ ]:
!cd /kaggle/working && tar czf laya_chat_finetuned.tgz laya_chat_finetuned && ls -lh laya_chat_finetuned.tgz


## 8.（可选）推到 Hugging Face Hub

In [ ]:
import os
from huggingface_hub import HfApi, login
# login(token=os.environ.get("HF_TOKEN"))
# HfApi().upload_folder(folder_path=OUTPUT_DIR, repo_id="<你的用户名>/laya-chat", repo_type="model")
